In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Define paths and load the dataset properly
DESKTOP = Path.home() / "Desktop"
DATA_DIR = DESKTOP / "demo data" / "raw"
csv = DATA_DIR / "customer_growth_potential.csv"

# Load the CSV file
df = pd.read_csv(csv)

def calculate_growth_potential(df):
    # Check if raw transaction columns exist, otherwise return the existing data safely
    required_cols = {'Customer', 'Date', '$ Spend'}
    if not required_cols.issubset(df.columns):
        print("Note: Raw transaction columns (Date, $ Spend) not found. Returning available customer summary.")
        if 'Customer' in df.columns and 'Customer Growth Potential' in df.columns:
            return df[['Customer', 'Customer Growth Potential']]
        return df

    # Sort data chronologically if columns are present
    df = df.sort_values(['Customer', 'Date'])
    
    # Calculate key metrics per customer
    customer_metrics = df.groupby('Customer').agg(
        total_purchases=('Date', 'count'),
        unique_years=('Date', 'nunique')
    ).reset_index()
    
    # Calculate year-over-year percentage growth for each transaction
    df['Pct_Growth'] = df.groupby('Customer')['$ Spend'].pct_change()
    
    # Check if a customer ever achieved >= 5% growth in their active periods
    growth_check = df.groupby('Customer')['Pct_Growth'].max().reset_index()
    growth_check['Has_5pct_Growth'] = growth_check['Pct_Growth'] >= 0.05
    
    # Merge metrics together
    summary = pd.merge(customer_metrics, growth_check[['Customer', 'Has_5pct_Growth']], on='Customer')
    
    # Apply classification rules
    conditions = [
        (summary['total_purchases'] == 1),
        (summary['unique_years'] == 3) & (summary['Has_5pct_Growth'] == True),
        (summary['unique_years'] == 3) & (summary['Has_5pct_Growth'] == False),
        (summary['total_purchases'] > 1) & (summary['Has_5pct_Growth'] == True),
        (summary['total_purchases'] > 1) & (summary['Has_5pct_Growth'] == False)
    ]
    choices = ['one time', 'loyal growth', 'loyal', 'regular growth', 'regular']
    summary['Customer Growth Potential'] = np.select(conditions, choices, default='unknown')
    
    return summary[['Customer', 'Customer Growth Potential']]

# Run the function
results = calculate_growth_potential(df)
print(results.to_string(index=False))

Note: Raw transaction columns (Date, $ Spend) not found. Returning available customer summary.
Customer Customer Growth Potential
       A              loyal growth
       B                   regular
       C            regular growth
       D                  one time


In [10]:
print(df.columns.tolist())

['Customer', 'Customer Growth Potential']
